# 零售教学数据生成：VIC 四种需求机制

本 Notebook 专门生成 CSV，供 Prophet、LightGBM 或其他模型共用。默认保留博客的日期、随机种子与公式，不进行拟合或模型排名。品牌名称仅说明教学情景，参数与营业规则均为作者假设。

运行顺序：安装依赖 → Run All → 检查 `data/teaching/` → 在建模 Notebook 中读取 CSV。

图片保存在：`images` 方便review，同时保留的 png 和 webp 格式（博客专用）。

默认日期为 2023-01-01 至 2025-12-31，共 1096 天；最后 28 天用于最终测试，截止日为 2025-12-03。生成器导出完整原始标签，训练集的标签屏蔽在建模阶段完成。

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import os
import numpy as np
import pandas as pd
import holidays


def find_repo_root():
    configured = os.environ.get("RETAIL_FORECAST_ROOT")
    candidates = (
        [Path(configured).expanduser()]
        if configured
        else [Path.cwd(), *Path.cwd().parents]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "notebooks"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT"
    )


REPO_ROOT = find_repo_root()
START_DATE = "2023-01-01"
END_DATE = "2025-12-31"
TIME_ORIGIN = pd.Timestamp("2023-01-01")  # 固定起点，决定季节相位与促销排期
HORIZON_DAYS = 28
OUTPUT_DIR = REPO_ROOT / "data" / "teaching"
IMAGE_DIR = REPO_ROOT / "images"

ALLOW_DATA_OVERWRITE = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)


def parse_ds_column(values, source):
    if values.isna().any():
        raise ValueError(f"{source} 的 ds 列包含空值")
    parsed = pd.to_datetime(values, format="%Y-%m-%d", errors="raise")
    canonical = parsed.dt.strftime("%Y-%m-%d")
    invalid = canonical.ne(values)
    if invalid.any():
        examples = values.loc[invalid].head(3).tolist()
        raise ValueError(
            f"{source} 的 ds 必须严格使用 yyyy-mm-dd 格式，异常值：{examples}"
        )
    return parsed


def read_teaching_csv(target):
    target = Path(target)
    frame = pd.read_csv(target, dtype={"ds": "string"})
    if "ds" not in frame.columns:
        raise ValueError(f"{target} 缺少必需的 ds 列")
    frame["ds"] = parse_ds_column(frame["ds"], str(target))
    return frame


def write_once(target, content):
    content = content if isinstance(content, str) else content.decode("utf-8")
    if (
        target.exists()
        and target.read_text(encoding="utf-8") != content
        and not ALLOW_DATA_OVERWRITE
    ):
        raise FileExistsError(
            f"{target} 已存在且内容不同；审阅后显式设置 ALLOW_DATA_OVERWRITE=True"
        )
    target.write_text(content, encoding="utf-8")


ds = pd.date_range(START_DATE, END_DATE, freq="D")
assert len(ds) > HORIZON_DAYS
# 使用固定起点；默认配置与原建模 Notebook 的 np.arange(len(ds)) 相同。
t = ((ds - TIME_ORIGIN) / pd.Timedelta(days=1)).to_numpy()
cutoff = ds[-HORIZON_DAYS - 1]
print({"days": len(ds), "cutoff": str(cutoff.date()), "test_days": HORIZON_DAYS})
display({name: version(name) for name in ["numpy", "pandas", "holidays"]})

## 1. 公共假日与营业日历

使用 `holidays` 的 AU/VIC 日历，并将同日多个事件拆成独立行，采用英文名称。导出的 `vic_holidays.csv` 包含 Prophet 所需的 `ds`、`holiday`、`lower_window`、`upper_window`。

公共假日表覆盖历史以及历史结束后 28 天涉及的年份。教学门店在 Good Friday 与 Christmas Day 闭店；其他日期按全天营业处理，ANZAC Day 未模拟营业时长缩短。四种方案沿用同一营业规则以方便对照；真实门店需要替换自己的日历与适用许可条件。

Cup Eve 与 New Year's Eve 是方案 2 的业务事件，通过独立标记登记。

In [ ]:
calendar_end = ds.max() + pd.Timedelta(days=HORIZON_DAYS)
calendar_years = list(range(ds.min().year, calendar_end.year + 1))
vic_calendar = holidays.country_holidays(
    "AU", subdiv="VIC", years=calendar_years, expand=False, language="en_US"
)
# get_list 按日历库的分隔规则拆出同一天的各个事件。
holiday_rows = [
    {"ds": pd.Timestamp(date), "holiday": name}
    for date in sorted(vic_calendar)
    for name in vic_calendar.get_list(date)
]
vic_holidays = (
    pd.DataFrame(holiday_rows).sort_values(["ds", "holiday"]).reset_index(drop=True)
)
vic_holidays["lower_window"] = 0
vic_holidays["upper_window"] = 0
closed_dates = pd.DatetimeIndex(
    vic_holidays.loc[
        vic_holidays["holiday"].isin(["Good Friday", "Christmas Day"]), "ds"
    ]
)
is_open = ~ds.isin(closed_dates)
is_vic_holiday = ds.isin(vic_holidays["ds"].unique()).astype(float)
assert (~is_open).sum() == ds.isin(closed_dates).sum()
display(vic_holidays)

## 2. 方案 1：商超高频刚需品（主案例）

$$\mu_t^{\mathrm{open}}=85+0.015t+16W_t+14\sin\left(\frac{2\pi(t-15)}{365.25}\right)+28H_t+130q_t$$
$$D_t^{\mathrm{sim}}=\max(0,\mu_t^{\mathrm{open}}+\epsilon_t),\qquad \epsilon_t\sim\mathcal N(0,6^2)$$
$$y_t^{\mathrm{sales}}=O_t\operatorname{round}\left[\min(D_t^{\mathrm{sim}},V_t)\right]$$

$t$ 是相对固定起点的日数，$W_t$ 是周末标记，$H_t$ 是公共假日标记，$q_t$ 是折扣比例（每 14 天的前 7 天为 0.20），$\mu_t^{\mathrm{open}}$ 是无噪声营业需求；$D_t^{\mathrm{sim}}$ 是含噪声潜在需求。$O_t$ 是营业标记，$V_t$ 在缺货时为 20 件，其余日期不设有限上限。$\epsilon_t$ 是独立正态噪声；$\mathcal N$ 表示正态分布，$\sim$ 表示服从，$\max$、$\min$ 分别取较大、较小值，$\operatorname{round}$ 表示按件取整。

营业日随机标记 3% 的缺货，标记与需求高低独立。此设定便于教学，真实缺货可能由高需求触发。闭店日仍保留假设营业的潜在需求，供诊断时使用。

In [ ]:
GROCERY_SEED = 42


def generate_grocery(seed=GROCERY_SEED):
    rng = np.random.default_rng(seed)
    discount = np.where(t % 14 < 7, 0.20, 0.0)
    weekend = (ds.dayofweek >= 5).astype(float)
    annual = 14 * np.sin(2 * np.pi * (t - 15) / 365.25)
    conditional_mean = np.maximum(
        0, 85 + 0.015 * t + 16 * weekend + annual + 130 * discount + 28 * is_vic_holiday
    )
    latent = np.maximum(0, conditional_mean + rng.normal(0, 6, len(ds)))
    stockout = (rng.random(len(ds)) < 0.03) & is_open
    sales = np.where(stockout, np.minimum(latent, 20), latent)
    return pd.DataFrame(
        {
            "ds": ds,
            "y": np.where(is_open, np.rint(sales), 0),
            "is_open": is_open,
            "is_closed_holiday": ~is_open,
            "discount": discount,
            "stockout": stockout,
            "conditional_mean": conditional_mean,
            "latent_demand": latent,
        }
    )


grocery_data = generate_grocery()
display(grocery_data.head())

import matplotlib.pyplot as plt

fig, (ax_all, ax_zoom) = plt.subplots(2, 1, figsize=(14, 8))
marker_styles = {
    "is_closed_holiday": ("s", "#D9534F", 48),
    "stockout": ("^", "#E69F00", 48),
}


def add_state_markers(axis, frame):
    for column, (marker, color, size) in marker_styles.items():
        mask = frame[column].astype(bool)
        axis.scatter(
            frame.loc[mask, "ds"],
            frame.loc[mask, "y"],
            marker=marker,
            color=color,
            s=size,
            alpha=0.75,
            label=f"{column}=True",
            zorder=3,
        )


ax_all.plot(
    grocery_data["ds"], grocery_data["y"], color="#2878B5", linewidth=0.8, label="y"
)
add_state_markers(ax_all, grocery_data)
ax_all.set_title("Synthetic grocery sales and operating-state markers")
ax_all.set_ylabel("observed sales (units)")
ax_all.legend(loc="upper left", ncol=3)
ax_all.grid(True, alpha=0.3)

zoom_start = pd.Timestamp("2023-03-18")
zoom_end = pd.Timestamp("2023-04-07")
zoom_data = grocery_data.loc[grocery_data["ds"].between(zoom_start, zoom_end)]
ax_zoom.plot(zoom_data["ds"], zoom_data["y"], color="#2878B5", label="y", zorder=2)
add_state_markers(ax_zoom, zoom_data)
ax_discount = ax_zoom.twinx()
ax_discount.plot(
    zoom_data["ds"],
    zoom_data["discount"],
    color="#C69200",
    marker="+",
    linestyle="--",
    label="discount",
)
ax_zoom.set_title("Zoom: Mar 18 - Apr 7, 2023, sales and discount")
ax_zoom.set_ylabel("observed sales (units)")
ax_discount.set_ylabel("discount")
ax_zoom.set_xlabel("ds")
handles, labels = ax_zoom.get_legend_handles_labels()
extra_handles, extra_labels = ax_discount.get_legend_handles_labels()
ax_zoom.legend(handles + extra_handles, labels + extra_labels, loc="upper right")
ax_zoom.grid(True, alpha=0.3)

fig.tight_layout()
for image_format in ("png", "webp"):
    fig.savefig(
        IMAGE_DIR / f"grocery-demand.{image_format}",
        dpi=150,
        bbox_inches="tight",
        format=image_format,
    )
plt.show()

## 3. 方案 2：单个红酒 SKU

$$\begin{aligned}
\lambda_t=\max\bigl(&10^{-6},4.5+0.002t+2.8I_{\mathrm{Thu},t}+7.5I_{\mathrm{FriSat},t}+2I_{\mathrm{Sun},t}\\
&+3.5\cos\left(\frac{2\pi(t-190)}{365.25}\right)+6I_{\mathrm{Dec},t}+12I_{\mathrm{party},t}+15C_t\bigr),\\
D_t^{\mathrm{wine}}&\sim\operatorname{Poisson}(\lambda_t).
\end{aligned}$$

$t$ 沿用固定起点日数，$\lambda_t$ 是泊松强度（采样均值与方差）。$I_{\mathrm{Thu},t}$、$I_{\mathrm{FriSat},t}$、$I_{\mathrm{Sun},t}$ 是星期标记；$I_{\mathrm{Dec},t}$ 标记 12 月 15—24 日；$I_{\mathrm{party},t}$ 标记 AFL 决赛前周五、墨尔本杯前夜及当天、跨年夜；$C_t$ 标记每 28 天的前三天成箱促销。$D_t^{\mathrm{wine}}$ 是潜在需求瓶数，$\operatorname{Poisson}$ 表示泊松分布，$10^{-6}$ 是正强度下界，$\max$ 取较大值，$\sim$ 表示服从。销售再由营业状态决定，闭店为零。

750 mL 单瓶口径。该配置不会自动构成高度间歇需求或零膨胀；读者可降低基线后重新测量营业日零值比例。

In [ ]:
WINE_SEED = 2027


def generate_wine(seed=WINE_SEED):
    rng = np.random.default_rng(seed)
    dow = ds.dayofweek
    december_peak = (ds.month == 12) & (ds.day >= 15) & (ds.day <= 24)
    cup_dates = pd.DatetimeIndex(
        vic_holidays.loc[
            vic_holidays.holiday.str.contains("Melbourne Cup", case=False), "ds"
        ]
    )
    afl_dates = pd.DatetimeIndex(
        vic_holidays.loc[
            vic_holidays.holiday.str.contains("Grand Final", case=False), "ds"
        ]
    )
    party = (
        ds.isin(cup_dates)
        | ds.isin(cup_dates - pd.Timedelta(days=1))
        | ds.isin(afl_dates)
        | ((ds.month == 12) & (ds.day == 31))
    )
    promo = (t % 28 < 3).astype(int)
    mean = (
        4.5
        + 0.002 * t
        + 2.8 * (dow == 3)
        + 7.5 * ((dow == 4) | (dow == 5))
        + 2 * (dow == 6)
    )
    mean += (
        3.5 * np.cos(2 * np.pi * (t - 190) / 365.25)
        + 6 * december_peak
        + 12 * party
        + 15 * promo
    )
    intensity = np.maximum(mean, 1e-6)
    latent = rng.poisson(intensity)
    return pd.DataFrame(
        {
            "ds": ds,
            "conditional_mean": intensity,
            "latent_demand": latent,
            "promo_case_buy": promo,
            "is_december_peak": december_peak.astype(int),
            "is_party_event": party.astype(int),
            "is_open": is_open,
            "y": np.where(is_open, np.rint(latent), 0),
            "stockout": False,
        }
    )


wine_data = generate_wine()
display(wine_data.head())

fig, (ax_all, ax_zoom) = plt.subplots(2, 1, figsize=(14, 8))
wine_marker_styles = {
    "is_closed_holiday": ("s", "#D9534F", 48),
    "stockout": ("^", "#E69F00", 48),
}


def add_wine_state_markers(axis, frame):
    closed = ~frame["is_open"]
    axis.scatter(
        frame.loc[closed, "ds"],
        frame.loc[closed, "y"],
        marker=wine_marker_styles["is_closed_holiday"][0],
        color=wine_marker_styles["is_closed_holiday"][1],
        s=wine_marker_styles["is_closed_holiday"][2],
        alpha=0.75,
        label="is_closed_holiday=True",
        zorder=3,
    )
    stockout = frame["stockout"].astype(bool)
    axis.scatter(
        frame.loc[stockout, "ds"],
        frame.loc[stockout, "y"],
        marker=wine_marker_styles["stockout"][0],
        color=wine_marker_styles["stockout"][1],
        s=wine_marker_styles["stockout"][2],
        alpha=0.75,
        label="stockout=True",
        zorder=3,
    )


ax_all.plot(wine_data["ds"], wine_data["y"], color="#2878B5", linewidth=0.8, label="y")
add_wine_state_markers(ax_all, wine_data)
ax_all.set_title("Synthetic wine sales and operating-state markers")
ax_all.set_ylabel("observed sales (bottles)")
ax_all.legend(loc="upper left", ncol=3)
ax_all.grid(True, alpha=0.3)

zoom_start = pd.Timestamp("2023-03-18")
zoom_end = pd.Timestamp("2023-04-07")
zoom_data = wine_data.loc[wine_data["ds"].between(zoom_start, zoom_end)]
ax_zoom.plot(zoom_data["ds"], zoom_data["y"], color="#2878B5", label="y", zorder=2)
add_wine_state_markers(ax_zoom, zoom_data)
ax_promo = ax_zoom.twinx()
ax_promo.plot(
    zoom_data["ds"],
    zoom_data["promo_case_buy"],
    color="#C69200",
    marker="o",
    linestyle="--",
    label="promo_case_buy",
)
ax_zoom.set_title("Zoom: Mar 18 - Apr 7, 2023, sales and case-buy promotion")
ax_zoom.set_ylabel("observed sales (bottles)")
ax_promo.set_ylabel("promo_case_buy")
ax_promo.set_yticks([0, 1], labels=["False", "True"])
ax_zoom.set_xlabel("ds")
handles, labels = ax_zoom.get_legend_handles_labels()
extra_handles, extra_labels = ax_promo.get_legend_handles_labels()
ax_zoom.legend(
    handles + extra_handles, labels + extra_labels, loc="upper right", ncol=3
)
ax_zoom.grid(True, alpha=0.3)

fig.tight_layout()
for image_format in ("png", "webp"):
    fig.savefig(
        IMAGE_DIR / f"wine-demand.{image_format}",
        dpi=150,
        bbox_inches="tight",
        format=image_format,
    )
plt.show()

## 4. 方案 3：秋季庭院耗材

$$\mu_t^{\mathrm{garden}}=50+0.01t+35I_{\mathrm{Sat},t}+25I_{\mathrm{Sun},t}+38\sin\left(\frac{2\pi(t-10)}{365.25}\right)+30L_t+12A_t$$
$$D_t^{\mathrm{garden}}=\max(0,\mu_t^{\mathrm{garden}}+\eta_t),\qquad\eta_t\sim\mathcal N(0,7^2).$$

$t$ 是日数，$I_{\mathrm{Sat},t}$、$I_{\mathrm{Sun},t}$ 为星期标记，$L_t$ 标记劳动节及前两天、复活节周六至周一，$A_t$ 标记 5 月 25—31 日清仓。$\mu_t^{\mathrm{garden}}$ 是无噪声营业需求，$D_t^{\mathrm{garden}}$ 是含噪声潜在需求，$\eta_t$ 是独立噪声；$\mathcal N(0,7^2)$ 表示均值 0、标准差 7 的正态分布，$\sim$ 表示服从，$\max$ 取较大值。清仓标记为 0/1，系数 12 对应增加 12 件。

教学门店沿用同一闭店规则。此数据未加入天气输入；独立随机噪声不代表模型已包含降雨效应。

In [ ]:
GARDEN_SEED = 2027


def generate_garden(seed=GARDEN_SEED):
    rng = np.random.default_rng(seed)
    dow = ds.dayofweek
    labour = pd.DatetimeIndex(
        vic_holidays.loc[
            vic_holidays.holiday.str.contains(r"Labou?r Day", regex=True), "ds"
        ]
    )
    easter = pd.DatetimeIndex(
        vic_holidays.loc[
            vic_holidays.holiday.isin(
                ["Easter Saturday", "Easter Sunday", "Easter Monday"]
            ),
            "ds",
        ]
    )
    diy_dates = (
        labour.union(labour - pd.Timedelta(days=1))
        .union(labour - pd.Timedelta(days=2))
        .union(easter)
    )
    long_weekend = ds.isin(diy_dates)
    clearance = ((ds.month == 5) & (ds.day >= 25)).astype(int)
    mean = (
        50
        + 0.01 * t
        + 35 * (dow == 5)
        + 25 * (dow == 6)
        + 38 * np.sin(2 * np.pi * (t - 10) / 365.25)
        + 30 * long_weekend
        + 12 * clearance
    )
    latent = np.maximum(0, mean + rng.normal(0, 7, len(ds)))
    return pd.DataFrame(
        {
            "ds": ds,
            "conditional_mean": mean,
            "latent_demand": latent,
            "is_long_weekend_diy": long_weekend.astype(int),
            "is_end_of_autumn_clearance": clearance,
            "is_open": is_open,
            "y": np.where(is_open, np.rint(latent), 0),
            "stockout": False,
        }
    )


garden_data = generate_garden()
display(garden_data.head())

fig, (ax_all, ax_zoom) = plt.subplots(2, 1, figsize=(14, 8))
ax_all.plot(
    garden_data["ds"], garden_data["y"], color="#2878B5", linewidth=0.8, label="y"
)
closed = ~garden_data["is_open"]
ax_all.scatter(
    garden_data.loc[closed, "ds"],
    garden_data.loc[closed, "y"],
    marker="s",
    color="#D9534F",
    s=48,
    alpha=0.75,
    label="is_closed_holiday=True",
    zorder=3,
)
stockout = garden_data["stockout"].astype(bool)
if stockout.any():
    ax_all.scatter(
        garden_data.loc[stockout, "ds"],
        garden_data.loc[stockout, "y"],
        marker="^",
        color="#E69F00",
        s=48,
        alpha=0.75,
        label="stockout=True",
        zorder=3,
    )
ax_all.set_title("Synthetic garden sales and operating-state markers")
ax_all.set_ylabel("observed sales (units)")
ax_all.legend(loc="upper left", ncol=3)
ax_all.grid(True, alpha=0.3)

zoom_start = pd.Timestamp("2023-05-18")
zoom_end = pd.Timestamp("2023-06-07")
zoom_data = garden_data.loc[garden_data["ds"].between(zoom_start, zoom_end)]
ax_zoom.plot(zoom_data["ds"], zoom_data["y"], color="#2878B5", label="y", zorder=2)
zoom_closed = ~zoom_data["is_open"]
ax_zoom.scatter(
    zoom_data.loc[zoom_closed, "ds"],
    zoom_data.loc[zoom_closed, "y"],
    marker="s",
    color="#D9534F",
    s=48,
    alpha=0.75,
    label="is_closed_holiday=True",
    zorder=3,
)
zoom_stockout = zoom_data["stockout"].astype(bool)
if zoom_stockout.any():
    ax_zoom.scatter(
        zoom_data.loc[zoom_stockout, "ds"],
        zoom_data.loc[zoom_stockout, "y"],
        marker="^",
        color="#E69F00",
        s=48,
        alpha=0.75,
        label="stockout=True",
        zorder=3,
    )
ax_event = ax_zoom.twinx()
ax_event.plot(
    zoom_data["ds"],
    zoom_data["is_end_of_autumn_clearance"],
    color="#C69200",
    marker="o",
    linestyle="--",
    label="is_end_of_autumn_clearance",
)
ax_zoom.set_title("Zoom: May 18-Jun 7, 2023, sales and autumn clearance")
ax_zoom.set_ylabel("observed sales (units)")
ax_event.set_ylabel("is_end_of_autumn_clearance")
ax_event.set_yticks([0, 1], labels=["False", "True"])
ax_zoom.set_xlabel("ds")
handles, labels = ax_zoom.get_legend_handles_labels()
extra_handles, extra_labels = ax_event.get_legend_handles_labels()
ax_zoom.legend(
    handles + extra_handles, labels + extra_labels, loc="upper right", ncol=3
)
ax_zoom.grid(True, alpha=0.3)

fig.tight_layout()
for image_format in ("png", "webp"):
    fig.savefig(
        IMAGE_DIR / f"garden-demand.{image_format}",
        dpi=150,
        bbox_inches="tight",
        format=image_format,
    )
plt.show()

## 4. 方案 4：奢侈品门店的间歇性高价值 SKU

以墨尔本 CBD 门店的一款高端手袋为例，将正需求发生与发生后的购买件数分开模拟。发生概率按周末、12 月旺季、VIP 预览活动和折扣强度使用 logistic 公式计算；发生后购买 1、2、5 或 10 件，概率分别为 0.75、0.15、0.08 和 0.02。该正需求量分布的理论均值为 1.65，理论 $CV_+^2$ 约为 0.97。

VIP 预览安排在每年 3、6、9、12 月首个周末，六月清仓活动设为 20% 折扣。教学门店沿用 Good Friday 与 Christmas Day 闭店规则，但不模拟缺货。`occurrence_probability`、`positive_demand`、`conditional_mean` 与 `latent_demand` 是仿真诊断真值，不应作为模型特征。

In [ ]:
LUXURY_SEED = 2085


def generate_luxury(seed=LUXURY_SEED):
    rng = np.random.default_rng(seed)
    is_saturday = ds.dayofweek == 5
    is_sunday = ds.dayofweek == 6
    is_december_peak = ds.month == 12

    vip_dates = []
    for year in sorted(ds.year.unique()):
        for month in (3, 6, 9, 12):
            first_week = pd.date_range(f"{year}-{month:02d}-01", periods=7, freq="D")
            first_saturday = first_week[first_week.dayofweek == 5][0]
            vip_dates.extend([first_saturday, first_saturday + pd.Timedelta(days=1)])
    is_vip_event = ds.isin(pd.DatetimeIndex(vip_dates))

    discount = np.where((ds.month == 6) & (ds.day >= 20) & (ds.day <= 26), 0.20, 0.0)
    logit_probability = (
        -3.2
        + 0.5 * is_saturday
        + 0.3 * is_sunday
        + 1.1 * is_december_peak
        + 1.4 * is_vip_event
        + 0.8 * discount
    )
    occurrence_probability = 1 / (1 + np.exp(-logit_probability))
    has_demand = rng.binomial(1, occurrence_probability).astype(bool)
    sampled_positive_demand = rng.choice(
        [1, 2, 5, 10], size=len(ds), p=[0.75, 0.15, 0.08, 0.02]
    )
    positive_demand = np.where(has_demand, sampled_positive_demand, np.nan)
    latent_demand = np.where(has_demand, sampled_positive_demand, 0)
    conditional_mean = occurrence_probability * 1.65
    stockout = np.zeros(len(ds), dtype=bool)

    return pd.DataFrame(
        {
            "ds": ds,
            "y": np.where(is_open, latent_demand, 0),
            "is_open": is_open,
            "is_closed_holiday": ~is_open,
            "is_saturday": is_saturday,
            "is_sunday": is_sunday,
            "is_december_peak": is_december_peak,
            "is_vip_event": is_vip_event,
            "discount": discount,
            "stockout": stockout,
            "occurrence_probability": occurrence_probability,
            "positive_demand": positive_demand,
            "conditional_mean": conditional_mean,
            "latent_demand": latent_demand,
        }
    )


luxury_data = generate_luxury()
opened_luxury = luxury_data.loc[luxury_data["is_open"]]
positive_sales = opened_luxury.loc[opened_luxury["y"] > 0, "y"]
luxury_adi = len(opened_luxury) / len(positive_sales)
luxury_cv_plus_sq = positive_sales.var(ddof=1) / positive_sales.mean() ** 2
print(
    {
        "open_days": len(opened_luxury),
        "positive_demand_days": len(positive_sales),
        "open_zero_rate": opened_luxury["y"].eq(0).mean(),
        "ADI": luxury_adi,
        "CV_plus_squared": luxury_cv_plus_sq,
        "theoretical_CV_plus_squared": 2.6275 / 1.65**2,
    }
)
display(luxury_data.head())

fig, (ax_all, ax_zoom) = plt.subplots(2, 1, figsize=(14, 8))
ax_all.step(
    luxury_data["ds"],
    luxury_data["y"],
    where="mid",
    color="#2878B5",
    linewidth=0.8,
    label="sales",
)
positive_mask = luxury_data["y"] > 0
ax_all.scatter(
    luxury_data.loc[positive_mask, "ds"],
    luxury_data.loc[positive_mask, "y"],
    marker="o",
    color="#2878B5",
    s=28,
    label="positive sales",
    zorder=3,
)
vip_mask = luxury_data["is_vip_event"]
ax_all.scatter(
    luxury_data.loc[vip_mask, "ds"],
    luxury_data.loc[vip_mask, "y"],
    marker="D",
    color="#D9534F",
    s=28,
    label="VIP event",
    zorder=4,
)
ax_all.set_title("Luxury handbag: intermittent daily sales")
ax_all.set_ylabel("sales (units)")
ax_all.legend(loc="upper left", ncol=3)
ax_all.grid(True, alpha=0.3)

zoom_start = pd.Timestamp("2023-12-01")
zoom_end = pd.Timestamp("2023-12-21")
zoom_data = luxury_data.loc[luxury_data["ds"].between(zoom_start, zoom_end)]
zoom_positive = zoom_data["y"] > 0
ax_zoom.step(
    zoom_data["ds"],
    zoom_data["y"],
    where="mid",
    color="#2878B5",
    label="sales",
    zorder=2,
)
ax_zoom.scatter(
    zoom_data.loc[zoom_positive, "ds"],
    zoom_data.loc[zoom_positive, "y"],
    marker="o",
    color="#2878B5",
    s=34,
    label="positive sales",
    zorder=3,
)
zoom_vip = zoom_data["is_vip_event"]
ax_zoom.scatter(
    zoom_data.loc[zoom_vip, "ds"],
    zoom_data.loc[zoom_vip, "y"],
    marker="D",
    color="#D9534F",
    s=36,
    label="VIP event",
    zorder=4,
)
ax_probability = ax_zoom.twinx()
ax_probability.plot(
    zoom_data["ds"],
    zoom_data["occurrence_probability"],
    color="#C69200",
    marker="o",
    linestyle="--",
    label="occurrence_probability",
)
ax_zoom.set_title("Zoom: Dec 1-21, 2023, sales and occurrence probability")
ax_zoom.set_ylabel("sales (units)")
ax_probability.set_ylabel("occurrence probability")
ax_zoom.set_xlabel("ds")
handles, labels = ax_zoom.get_legend_handles_labels()
extra_handles, extra_labels = ax_probability.get_legend_handles_labels()
ax_zoom.legend(handles + extra_handles, labels + extra_labels, loc="upper left", ncol=2)
ax_zoom.grid(True, alpha=0.3)

fig.tight_layout()
for image_format in ("png", "webp"):
    fig.savefig(
        IMAGE_DIR / f"luxury-handbag-demand.{image_format}",
        dpi=150,
        bbox_inches="tight",
        format=image_format,
    )
plt.show()

## 5. 验证数据并导出 CSV

每个方案都保留 `ds`、`y`、`is_open`、`stockout`。`conditional_mean` 和 `latent_demand` 是用于诊断的数据生成真实值，只用于核对规律；请在建模时显式选择输入列。

`y` 是原始销售，闭店为 0，缺货时受限。CSV 中布尔字段写为 `True`/`False`，用 pandas 读回时检查字段类型。`vic_holidays.csv` 可直接交给 Prophet 的 `holidays=`。

同一固定配置重复运行将覆盖同名生成文件。若改变日期、种子或公式，建议调整输出目录，并重新运行所有模型与评估，更新博客数值。

In [ ]:
datasets = {
    "grocery": grocery_data,
    "wine": wine_data,
    "garden": garden_data,
    "luxury": luxury_data,
}
regressor_columns = {
    "grocery": ["discount"],
    "wine": ["promo_case_buy", "is_december_peak", "is_party_event"],
    "garden": ["is_end_of_autumn_clearance", "is_long_weekend_diy"],
    "luxury": [
        "is_saturday",
        "is_sunday",
        "is_december_peak",
        "is_vip_event",
        "discount",
    ],
}
summary = []
for name, frame in datasets.items():
    assert frame.ds.is_unique and frame.ds.is_monotonic_increasing
    assert len(frame) == len(ds) and frame.ds.equals(pd.Series(ds, name="ds"))
    assert frame.y.notna().all() and (frame.y >= 0).all()
    assert (frame.y == np.rint(frame.y)).all()
    assert (frame.loc[~frame.is_open, "y"] == 0).all()
    assert not (frame.stockout & ~frame.is_open).any()
    assert np.isfinite(frame.conditional_mean).all()
    assert np.isfinite(frame.latent_demand).all()
    assert (frame.ds > cutoff).sum() == HORIZON_DAYS

    opened = frame.loc[frame.is_open]
    positive = opened.loc[opened.y > 0, "y"]
    positive_days = len(positive)
    adi = len(opened) / positive_days if positive_days else np.nan
    cv_plus_squared = (
        positive.var(ddof=1) / positive.mean() ** 2 if positive_days > 1 else np.nan
    )

    regenerated = {
        "grocery": generate_grocery,
        "wine": generate_wine,
        "garden": generate_garden,
        "luxury": generate_luxury,
    }[name]()
    pd.testing.assert_frame_equal(frame, regenerated)
    target = OUTPUT_DIR / f"{name}_scenario_data.csv"
    write_once(target, frame.to_csv(index=False, date_format="%Y-%m-%d"))
    reloaded = read_teaching_csv(target)
    pd.testing.assert_frame_equal(
        frame, reloaded, check_exact=False, rtol=1e-12, atol=1e-12
    )

    summary.append(
        {
            "scenario": name,
            "rows": len(frame),
            "closed_days": int((~frame.is_open).sum()),
            "stockout_days": int(frame.stockout.sum()),
            "open_mean_sales": opened.y.mean(),
            "open_zero_rate": opened.y.eq(0).mean(),
            "positive_demand_days": positive_days,
            "ADI": adi,
            "CV_plus_squared": cv_plus_squared,
            "open_min_conditional_mean": opened.conditional_mean.min(),
            "final_test_days": int((frame.ds > cutoff).sum()),
        }
    )

write_once(
    OUTPUT_DIR / "vic_holidays.csv",
    vic_holidays.to_csv(index=False, date_format="%Y-%m-%d"),
)
diagnostics = pd.DataFrame(summary)
diagnostics_path = OUTPUT_DIR / "scenario_diagnostics.csv"
diagnostics.to_csv(diagnostics_path, index=False)
metadata = {
    "start_date": START_DATE,
    "end_date": END_DATE,
    "time_origin": str(TIME_ORIGIN.date()),
    "horizon_days": HORIZON_DAYS,
    "cutoff": str(cutoff.date()),
    "diagnostics_file": diagnostics_path.name,
    "seeds": {
        "grocery": GROCERY_SEED,
        "wine": WINE_SEED,
        "garden": GARDEN_SEED,
        "luxury": LUXURY_SEED,
    },
    "versions": {name: version(name) for name in ["numpy", "pandas", "holidays"]},
    "closed_holiday_names": ["Good Friday", "Christmas Day"],
    "regressor_columns": regressor_columns,
    "diagnostic_only_columns": [
        "conditional_mean",
        "occurrence_probability",
        "positive_demand",
        "latent_demand",
    ],
    "csv_sha256": {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(OUTPUT_DIR.glob("*.csv"))
    },
}
metadata_path = OUTPUT_DIR / "generation_metadata.json"
metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(diagnostics)
print("诊断表：", str(diagnostics_path.resolve()))
print("元数据：", str(metadata_path.resolve()))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

frequent_scenarios = diagnostics[
    diagnostics["scenario"].isin(["grocery", "wine", "garden"])
]
luxury_scenario = diagnostics[diagnostics["scenario"] == "luxury"]
features = diagnostics[["ADI", "CV_plus_squared"]].to_numpy(dtype=float)
labels = diagnostics["scenario"].eq("luxury").astype(int).to_numpy()
svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear", C=10.0, class_weight="balanced", random_state=0),
)
svm.fit(features, labels)
assert np.array_equal(svm.predict(features), labels)

scaler = svm.named_steps["standardscaler"]
classifier = svm.named_steps["svc"]
raw_coefficients = classifier.coef_[0] / scaler.scale_
raw_intercept = classifier.intercept_[0] - np.dot(raw_coefficients, scaler.mean_)
svm_slope = -raw_coefficients[0] / raw_coefficients[1]
svm_intercept = -raw_intercept / raw_coefficients[1]
svm_equation = (
    rf"$\mathrm{{CV}}_+^2 = {svm_slope:.4f}\,\mathrm{{ADI}} {svm_intercept:+.4f}$"
)

fig, ax = plt.subplots(figsize=(7, 6))
adi_limit = max(17.0, diagnostics["ADI"].max() * 1.1)
cv_limit = max(1.1, diagnostics["CV_plus_squared"].max() * 1.1)
grid_adi, grid_cv = np.meshgrid(
    np.linspace(0, adi_limit, 300),
    np.linspace(0, cv_limit, 300),
)
grid_features = np.column_stack([grid_adi.ravel(), grid_cv.ravel()])
decision_values = svm.decision_function(grid_features).reshape(grid_adi.shape)
decision_limit = np.abs(decision_values).max()
decision_norm = TwoSlopeNorm(vmin=-decision_limit, vcenter=0, vmax=decision_limit)
decision_cmap = LinearSegmentedColormap.from_list(
    "svm_decision_regions", ["#BFDCEC", "#F7F8FA", "#F1C3C6"]
)
heatmap = ax.contourf(
    grid_adi,
    grid_cv,
    decision_values,
    levels=41,
    cmap=decision_cmap,
    norm=decision_norm,
    alpha=0.8,
    extend="both",
    zorder=0,
)

ax.scatter(
    frequent_scenarios["ADI"],
    frequent_scenarios["CV_plus_squared"],
    color="#2878B5",
    marker="o",
    s=75,
    label="Grocery / wine / garden",
    zorder=3,
)
ax.scatter(
    luxury_scenario["ADI"],
    luxury_scenario["CV_plus_squared"],
    color="#D9534F",
    marker="D",
    s=90,
    label="Luxury handbag",
    zorder=4,
)

scenario_labels = {
    "grocery": "Grocery",
    "wine": "Wine",
    "garden": "Garden",
    "luxury": "Luxury",
}
label_offsets = {
    "grocery": (8, -12),
    "wine": (8, 6),
    "garden": (8, -2),
    "luxury": (8, 0),
}
for row in diagnostics.itertuples(index=False):
    ax.annotate(
        scenario_labels[row.scenario],
        (row.ADI, row.CV_plus_squared),
        xytext=label_offsets[row.scenario],
        textcoords="offset points",
    )

ax.contour(
    grid_adi,
    grid_cv,
    decision_values,
    levels=[0],
    colors=["#6A4C93"],
    linestyles=["-."],
    linewidths=1.7,
)
ax.axvline(1.32, color="0.4", linestyle="--", linewidth=1, label="ADI reference: 1.32")
ax.axhline(
    0.49, color="0.55", linestyle=":", linewidth=1, label="CV+ squared reference: 0.49"
)
ax.text(
    0.03,
    0.95,
    "Frequent / lower variability",
    transform=ax.transAxes,
    va="top",
    color="#2878B5",
)
ax.text(
    0.68,
    0.95,
    "Intermittent / lumpy",
    transform=ax.transAxes,
    va="top",
    color="#D9534F",
)

formula_x = 8.0
formula_y = svm_slope * formula_x + svm_intercept
ax.annotate(
    svm_equation,
    xy=(formula_x, formula_y),
    xytext=(formula_x - 1.8, formula_y + 0.13),
    textcoords="data",
    ha="center",
    va="bottom",
    fontsize=9,
    bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#6A4C93", "alpha": 0.95},
    arrowprops={"arrowstyle": "-", "color": "#6A4C93", "linewidth": 0.9},
)

ax.set_xlim(0, adi_limit)
ax.set_ylim(0, cv_limit)
ax.set_xlabel("ADI (average demand interval)")
ax.set_ylabel("CV+ squared (positive-demand variability)")
ax.set_title("Retail scenarios: demand profiles")
ax.grid(True, alpha=0.25)
handles, legend_labels = ax.get_legend_handles_labels()
handles.append(
    Line2D(
        [],
        [],
        color="#6A4C93",
        linestyle="-.",
        label="Linear SVM boundary (illustrative, n=4)",
    )
)
ax.legend(handles=handles, loc="center right")
colorbar = fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
colorbar.set_label("SVM decision score (not probability)")
fig.tight_layout()
for image_format in ("png", "webp"):
    fig.savefig(
        IMAGE_DIR / f"retail-demand-profile-adi-cv.{image_format}",
        dpi=150,
        bbox_inches="tight",
        format=image_format,
    )
plt.show()

## 6. 在建模 Notebook 中读取并检查场景数据

下面读取方案 1 主案例并按预测截止日划分训练集与测试集；同时读取方案 2、3、4 的 CSV，检查日期、目标销量和行数，并输出各场景的数值列描述统计。`conditional_mean`、`occurrence_probability`、`positive_demand` 和 `latent_demand` 属于合成数据诊断真值，不能作为模型特征。

In [ ]:
data = read_teaching_csv(OUTPUT_DIR / "grocery_scenario_data.csv")
vic_holidays = read_teaching_csv(OUTPUT_DIR / "vic_holidays.csv")
for col in ["is_open", "is_closed_holiday", "stockout"]:
    assert pd.api.types.is_bool_dtype(data[col]), f"请检查布尔字段：{col}"
cutoff = data["ds"].iloc[-HORIZON_DAYS - 1]
train = data.loc[data.ds <= cutoff].copy()
test = data.loc[data.ds > cutoff].copy()
train.loc[train.stockout | ~train.is_open, "y"] = np.nan
assert len(test) == HORIZON_DAYS and train.ds.max() < test.ds.min()
# 重建后续主案例使用的变量；建模输入显式选择 ds、y、discount。
ds = pd.DatetimeIndex(data["ds"])
t = ((ds - TIME_ORIGIN) / pd.Timedelta(days=1)).to_numpy()
rng = np.random.default_rng(GROCERY_SEED)


def sales_from_open_forecast(frame, raw_prediction):
    open_mask = frame["is_open"].to_numpy(bool)
    return np.where(open_mask, np.maximum(np.asarray(raw_prediction, float), 0), 0)


print(
    {
        "training_rows": len(train),
        "valid_training_labels": int(train.y.notna().sum()),
        "test_rows": len(test),
        "training_y_max": float(train.y.max()),
    }
)
print("方案 1：商超高频刚需品 data.describe()")
display(data.describe(include="all").T)

scenario_files = {
    "方案 2：红酒 SKU": "wine_scenario_data.csv",
    "方案 3：庭院耗材": "garden_scenario_data.csv",
    "方案 4：奢侈品手袋": "luxury_scenario_data.csv",
}
scenario_data = {}
for scenario_name, file_name in scenario_files.items():
    scenario_frame = read_teaching_csv(OUTPUT_DIR / file_name)
    assert scenario_frame.ds.is_unique and scenario_frame.ds.is_monotonic_increasing
    assert len(scenario_frame) == len(data)
    assert scenario_frame.y.notna().all() and (scenario_frame.y >= 0).all()
    assert scenario_frame.loc[~scenario_frame.is_open, "y"].eq(0).all()
    assert not (scenario_frame.stockout & ~scenario_frame.is_open).any()
    scenario_data[scenario_name] = scenario_frame
    print(f"{scenario_name}: {len(scenario_frame)} rows")
    display(scenario_frame.describe(include="all").T)